# 01 · Data Cleaning & PostgreSQL Load
**Project:** Reel Insights – TMDB Movie Industry EDA

**Goal:** load the six raw TMDB CSVs, clean them (nulls, duplicates, types, referential integrity), save cleaned copies, and load them into a PostgreSQL database (`tmdb`) with primary/foreign keys.

| Table | Grain | Key |
|---|---|---|
| movies | 1 row per movie | movie_id |
| genres | 1 row per genre | genre_id |
| movie_genres | movie × genre bridge | (movie_id, genre_id) |
| cast | movie × actor credit | cast_row_id |
| crew | movie × crew credit | crew_row_id |
| movie_keywords | movie × keyword | (movie_id, keyword_id) |

## 1. Setup

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import text
from db_config import get_engine, DB_URL   # connection settings live in db_config.py

pd.set_option("display.max_columns", 50)
RAW = Path("data")            # six raw CSVs
CLEAN = Path("data/clean")    # cleaned output
CLEAN.mkdir(parents=True, exist_ok=True)
TABLES = ["movies", "genres", "movie_genres", "cast", "crew", "movie_keywords"]

## 2. Load raw CSVs

In [15]:
# Read every CSV into a dict of DataFrames
raw = {t: pd.read_csv(RAW / f"{t}.csv") for t in TABLES}
for t, df in raw.items():
    print(f"{t:<15} {df.shape[0]:>7,} rows  x {df.shape[1]} cols")

movies            2,509 rows  x 12 cols
genres               19 rows  x 2 cols
movie_genres      6,888 rows  x 2 cols
cast             24,917 rows  x 6 cols
crew              7,297 rows  x 6 cols
movie_keywords   41,701 rows  x 3 cols


## 3. Profile: data types, nulls, duplicates

In [16]:
# One summary row per table: nulls and fully-duplicated rows
profile = pd.DataFrame({
    t: {"rows": len(df), "null_cells": int(df.isna().sum().sum()), "duplicate_rows": int(df.duplicated().sum())}
    for t, df in raw.items()
}).T
profile

,rows,null_cells,duplicate_rows
movies,2509,0,6
genres,19,0,0
movie_genres,6888,0,10
cast,24917,11,15
crew,7297,0,8
movie_keywords,41701,0,20


In [17]:
# Column-level nulls for the main fact table
raw["movies"].info()
raw["movies"].isna().sum().to_frame("nulls").query("nulls > 0")

<class 'pandas.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2509 non-null   int64  
 1   title              2509 non-null   str    
 2   original_language  2509 non-null   str    
 3   release_date       2509 non-null   str    
 4   runtime            2509 non-null   int64  
 5   budget             2509 non-null   float64
 6   revenue            2509 non-null   float64
 7   popularity         2509 non-null   float64
 8   vote_average       2509 non-null   float64
 9   vote_count         2509 non-null   int64  
 10  status             2509 non-null   str    
 11  overview           2509 non-null   str    
dtypes: float64(4), int64(3), str(5)
memory usage: 989.6 KB


,nulls


In [18]:
# TMDB stores UNKNOWN money/runtime as 0 (not free / not 0 minutes)
m = raw["movies"]
print("budget == 0 :", (m["budget"] == 0).sum())
print("revenue == 0:", (m["revenue"] == 0).sum())
print("runtime == 0:", (m["runtime"] == 0).sum())

budget == 0 : 90
revenue == 0: 102
runtime == 0: 0


**Observation:** the tables contain exact duplicate rows, some null dates/runtimes, and many `0` values in budget/revenue.

**Cleaning decisions**
1. Drop exact duplicate rows, then keep one row per primary key.
2. Trim whitespace on text columns; enforce numeric and date types.
3. `runtime = 0` → NULL (a film cannot run 0 minutes).
4. `budget` / `revenue = 0` are **kept as 0 in the database** so the raw fact is preserved. They are treated as *unknown* in analysis (`NULLIF(x,0)` in SQL, `replace(0, NaN)` in Pandas – see notebook 03, Q2).
5. Remove child rows whose `movie_id` / `genre_id` does not exist in the parent table (orphans), so foreign keys can be enforced.

## 4. Clean each table

In [19]:
def strip_text(df):
    """Trim whitespace in every text column and turn empty strings into NaN."""
    df = df.copy()
    for c in [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]:
        df[c] = df[c].str.strip().replace("", np.nan)
    return df

# ---- movies ----
movies = strip_text(raw["movies"]).drop_duplicates()
movies = movies.dropna(subset=["movie_id"]).drop_duplicates(subset="movie_id", keep="first")
movies["movie_id"] = movies["movie_id"].astype(int)
for c in ["runtime", "budget", "revenue", "popularity", "vote_average", "vote_count"]:
    movies[c] = pd.to_numeric(movies[c], errors="coerce")
movies["budget"] = movies["budget"].fillna(0).astype("int64")
movies["revenue"] = movies["revenue"].fillna(0).astype("int64")
movies["vote_count"] = movies["vote_count"].fillna(0).astype("int64")
movies["runtime"] = movies["runtime"].replace(0, np.nan)          # 0 min = unknown
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce").dt.date
movies = movies.dropna(subset=["title"])

# ---- genres ----
genres = strip_text(raw["genres"]).drop_duplicates().drop_duplicates(subset="genre_id")

valid_movies = set(movies["movie_id"])
valid_genres = set(genres["genre_id"])

# ---- bridge / child tables: dedupe + drop orphans ----
movie_genres = raw["movie_genres"].drop_duplicates().dropna()
movie_genres = movie_genres[movie_genres["movie_id"].isin(valid_movies) & movie_genres["genre_id"].isin(valid_genres)]

cast = strip_text(raw["cast"]).drop_duplicates().drop_duplicates(subset="cast_row_id")
cast = cast.dropna(subset=["actor_name"])
cast = cast[cast["movie_id"].isin(valid_movies)]

crew = strip_text(raw["crew"]).drop_duplicates().drop_duplicates(subset="crew_row_id")
crew = crew.dropna(subset=["person_name", "job"])
crew = crew[crew["movie_id"].isin(valid_movies)]

movie_keywords = strip_text(raw["movie_keywords"]).drop_duplicates(subset=["movie_id", "keyword_id"]).dropna(subset=["keyword_name"])
movie_keywords = movie_keywords[movie_keywords["movie_id"].isin(valid_movies)]

clean = dict(movies=movies, genres=genres, movie_genres=movie_genres,
             cast=cast, crew=crew, movie_keywords=movie_keywords)

In [20]:
# Before vs after row counts
pd.DataFrame({"raw_rows": {t: len(raw[t]) for t in TABLES},
              "clean_rows": {t: len(clean[t]) for t in TABLES}}).assign(
    removed=lambda d: d.raw_rows - d.clean_rows)

,raw_rows,clean_rows,removed
movies,2509,2503,6
genres,19,19,0
movie_genres,6888,6878,10
cast,24917,24902,15
crew,7297,7289,8
movie_keywords,41701,41681,20


In [21]:
# Save cleaned CSVs (used as a fallback by the Streamlit app)
for t, df in clean.items():
    df.to_csv(CLEAN / f"{t}.csv", index=False)
print("Saved to", CLEAN.resolve())

Saved to /Users/kamila/Desktop/TMDB_Reel_Insights/data/clean


## 5. Create PostgreSQL schema
Tables are created with explicit types, primary keys and foreign keys. The DDL is also saved to `data/tmdb_schema.sql` so the database can be rebuilt from pgAdmin (Query Tool → open file → run).

In [22]:
DDL = '''
DROP TABLE IF EXISTS movie_keywords, crew, "cast", movie_genres, genres, movies CASCADE;

CREATE TABLE movies (
    movie_id          INTEGER PRIMARY KEY,
    title             TEXT NOT NULL,
    original_language VARCHAR(10),
    release_date      DATE,
    runtime           NUMERIC,
    budget            BIGINT DEFAULT 0,
    revenue           BIGINT DEFAULT 0,
    popularity        NUMERIC,
    vote_average      NUMERIC(3,1),
    vote_count        INTEGER,
    status            VARCHAR(30),
    overview          TEXT
);
CREATE TABLE genres (
    genre_id   INTEGER PRIMARY KEY,
    genre_name VARCHAR(50) NOT NULL
);
CREATE TABLE movie_genres (
    movie_id INTEGER REFERENCES movies(movie_id),
    genre_id INTEGER REFERENCES genres(genre_id),
    PRIMARY KEY (movie_id, genre_id)
);
CREATE TABLE "cast" (
    cast_row_id    INTEGER PRIMARY KEY,
    movie_id       INTEGER REFERENCES movies(movie_id),
    person_id      INTEGER,
    actor_name     TEXT NOT NULL,
    character_name TEXT,
    cast_order     INTEGER
);
CREATE TABLE crew (
    crew_row_id INTEGER PRIMARY KEY,
    movie_id    INTEGER REFERENCES movies(movie_id),
    person_id   INTEGER,
    person_name TEXT NOT NULL,
    job         VARCHAR(100),
    department  VARCHAR(100)
);
CREATE TABLE movie_keywords (
    movie_id     INTEGER REFERENCES movies(movie_id),
    keyword_id   INTEGER,
    keyword_name TEXT,
    PRIMARY KEY (movie_id, keyword_id)
);
CREATE INDEX idx_cast_movie  ON "cast"(movie_id);
CREATE INDEX idx_crew_movie  ON crew(movie_id);
CREATE INDEX idx_crew_job    ON crew(job);
CREATE INDEX idx_mk_movie    ON movie_keywords(movie_id);
'''
Path("data/tmdb_schema.sql").write_text(DDL)

engine = get_engine()
with engine.begin() as conn:
    conn.execute(text(DDL))
print("Schema created in", DB_URL.split("@")[-1])

Schema created in localhost:5432/tmdb


## 6. Load data (parents first, then children)

In [23]:
# Order matters because of foreign keys
with engine.begin() as conn:
    for t in TABLES:
        clean[t].to_sql(t, conn, if_exists="append", index=False, method="multi", chunksize=2000)
        print(f"loaded {t:<15} {len(clean[t]):>7,} rows")

loaded movies            2,503 rows
loaded genres               19 rows
loaded movie_genres      6,878 rows
loaded cast             24,902 rows
loaded crew              7,289 rows
loaded movie_keywords   41,681 rows


## 7. Validate the load

In [24]:
# Row counts in PostgreSQL must equal the cleaned DataFrames
with engine.connect() as conn:
    check = {t: conn.execute(text(f'SELECT COUNT(*) FROM "{t}"')).scalar() for t in TABLES}
val = pd.DataFrame({"dataframe": {t: len(clean[t]) for t in TABLES}, "postgres": check})
val["match"] = val.dataframe == val.postgres
assert val["match"].all(), "Row count mismatch!"
val

,dataframe,postgres,match
movies,2503,2503,True
genres,19,19,True
movie_genres,6878,6878,True
cast,24902,24902,True
crew,7289,7289,True
movie_keywords,41681,41681,True


## 8. Summary & insight
- All six tables were cleaned and loaded into PostgreSQL with primary and foreign keys enforced – any orphan credit/genre rows were removed before load.
- Duplicate movie rows were removed so each `movie_id` is counted once; this prevents inflated revenue totals in later analysis.
- A large share of movies carry `0` budget/revenue. These are **unknown**, not zero – every financial metric downstream excludes them, otherwise averages and ROI would be badly understated.

**Recommendation:** treat budget/revenue coverage as a data-quality KPI. Any business decision based on financials (ROI, genre economics) should state the sample size of movies with valid figures.